# Transient cylinder wake

**Unverified product example.** This demonstrates the public transient workflow and makes no benchmark, convergence, drag/lift, or shedding claim. In Google Colab, the first code cell reconciles the pinned `0.1.0` release while retaining a coherent supported preloaded Matplotlib; it does not use or create maintainer-owned Drive state.


In [ ]:
from ctypes.util import find_library
from importlib.metadata import distribution, version
from importlib.resources import files
from importlib.util import find_spec
import os
from pathlib import Path
import re
import signal
import subprocess
import sys

EQIORA_VERSION = "0.1.0"
GMSH_VERSION = "4.15.2"
MATPLOTLIB_RANGE = ">=3.10,<3.12"
_MODULE_DISTRIBUTION_FILES = {
    "eqiora": ("eqiora", "eqiora/__init__.py"),
    "matplotlib": ("matplotlib", "matplotlib/__init__.py"),
    "matplotlib._api": ("matplotlib", "matplotlib/_api/__init__.py"),
    "mpl_toolkits.axes_grid1.axes_divider": (
        "matplotlib",
        "mpl_toolkits/axes_grid1/axes_divider.py",
    ),
}


def _module_state():
    return {
        name: {
            "version": getattr(module, "__version__", None),
            "file": getattr(module, "__file__", None),
        }
        for name in _MODULE_DISTRIBUTION_FILES
        if (module := sys.modules.get(name)) is not None
    }


def _distribution_state():
    packages = {
        owner: distribution(owner)
        for owner in {
            owner for owner, _relative in _MODULE_DISTRIBUTION_FILES.values()
        }
    }
    return {
        name: {
            "version": packages[owner].version,
            "file": str(
                Path(
                    packages[owner].locate_file(relative)
                ).resolve()
            ),
        }
        for name, (owner, relative) in _MODULE_DISTRIBUTION_FILES.items()
    }


def _environment_is_coherent(loaded, installed):
    release = tuple(
        int(part)
        for part in re.findall(r"[0-9]+", installed["matplotlib"]["version"])[:2]
    )
    if len(release) != 2 or not (release >= (3, 10) and release < (3, 12)):
        return False
    return all(
        state["file"] is not None
        and str(Path(state["file"]).resolve()) == installed[name]["file"]
        and (state["version"] is None or state["version"] == installed[name]["version"])
        for name, state in loaded.items()
    )


def _prepare_environment():
    loaded = _module_state()
    if find_spec("google.colab") is not None and find_library("GLU") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(
            ["apt-get", "install", "-y", "-qq", "libglu1-mesa"],
            check=True,
        )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade-strategy",
            "only-if-needed",
            f"eqiora=={EQIORA_VERSION}",
            f"gmsh=={GMSH_VERSION}",
            f"matplotlib{MATPLOTLIB_RANGE}",
        ],
        check=True,
    )
    if version("eqiora") != EQIORA_VERSION or version("gmsh") != GMSH_VERSION:
        raise RuntimeError("pip did not select the requested Eqiora and Gmsh releases")
    installed = _distribution_state()
    if not _environment_is_coherent(loaded, installed):
        print({"loaded_before_install": loaded, "installed_after_install": installed})
        if find_spec("google.colab") is None:
            raise RuntimeError("installed packages changed under loaded modules; restart Python")
        os.kill(os.getpid(), signal.SIGKILL)
        raise RuntimeError("Colab restart did not terminate the runtime")
    return installed


_installed_modules = _prepare_environment()

import eqiora
import eqiora.matplotlib as eqplot
import matplotlib
import matplotlib._api
import numpy as np

print(
    {
        "matplotlib.__version__": matplotlib.__version__,
        "matplotlib.__file__": matplotlib.__file__,
        "matplotlib._api.__file__": matplotlib._api.__file__,
        "mpl_toolkits.axes_grid1.axes_divider": _installed_modules[
            "mpl_toolkits.axes_grid1.axes_divider"
        ]["file"],
    }
)


In [ ]:
geometry_graph = eqiora.geometry.GeometryGraph()
rectangle = geometry_graph.rectangle(x_bounds=(0.0, 2.2), y_bounds=(0.0, 0.41))
circle = geometry_graph.circle(center=(0.2, 0.2), radius=0.05)
fluid = geometry_graph.subtract(rectangle, circle)
geometry = geometry_graph.build(
    fluid,
    named_topology={
        "fluid": fluid.region,
        "inlet": rectangle.boundaries[0],
        "outlet": rectangle.boundaries[1],
        "walls": rectangle.boundaries[2:4],
        "cylinder": circle.boundaries[0],
    },
)
geometry

In [ ]:
mesh_request = eqiora.meshing.GmshMesher(
    maximum_boundary_error=1.0e-4,
    maximum_target_size=0.025,
    minimum_mean_ratio=1.0e-5,
    maximum_boundary_facets=50,
)
mesh_plan = eqiora.meshing.resolve(geometry, mesh_request)
mesh = eqiora.meshing.generate(mesh_plan)
mesh

In [ ]:
source_root = files(eqiora).joinpath("examples")
parameters = {
    "dynamic_viscosity": 1.0e-3,
    "zero_pressure": 0.0,
    "inlet_speed": 0.3,
    "channel_height": geometry.bounds[1][1] - geometry.bounds[1][0],
}
support_bindings = {
    "fluid": geometry.selection("fluid"),
    **{
        side: (geometry.selection(side), geometry.selection("fluid"))
        for side in ("inlet", "outlet", "walls", "cylinder")
    },
}
steady_model = eqiora.compile(
    path=source_root.joinpath("steady-flow-past-cylinder.eqi"),
    geometry=geometry,
    entry="SteadyFlowPastCylinder",
    bindings={**support_bindings, **parameters},
)
linear = eqiora.solve.Linear(
    algorithm=eqiora.solve.LinearSolver.SparseLu,
    preconditioner=eqiora.solve.Preconditioner.Identity,
    reduction=eqiora.solve.Reduction.Fast,
    provider=eqiora.solve.SolverProvider.faer(),
    relative_tolerance=1.0e-6,
    absolute_tolerance=1.0e-9,
    maximum_iterations=20_000,
)
steady_plan = eqiora.resolve(
    steady_model,
    mesh=mesh,
    spatial=eqiora.fem.MiniP1(),
    solve=linear,
    scaling=None,
)
steady_result = eqiora.run(steady_plan)
steady_result

In [ ]:
model = eqiora.compile(
    path=source_root.joinpath("transient-flow-past-cylinder.eqi"),
    geometry=geometry,
    entry="TransientFlowPastCylinder",
    bindings={**support_bindings, "density": 1.0, **parameters},
)
plan = eqiora.resolve(
    model,
    mesh=mesh,
    spatial=eqiora.fem.MiniP1(),
    temporal=eqiora.time.BackwardEuler(0.01),
    solve=eqiora.solve.Newton(linear=linear),
    scaling=eqiora.fluid.IncompressibleScaling(
        length_m=0.41,
        velocity_m_per_s=0.3,
        pressure_pa=0.09,
    ),
)
plan

In [ ]:
steady_velocity = steady_result.output(steady_plan.capability.velocity)
steady_pressure = steady_result.output(steady_plan.capability.pressure)
state = eqiora.State.initial(
    plan,
    time_s=0.0,
    fields=(
        eqiora.InitialField(
            plan.capability.velocity,
            vertex_values=np.asarray(steady_velocity.values("vertex")).reshape(
                mesh.vertex_count, 2
            ),
            cell_values=np.asarray(steady_velocity.values("cell-bubble")).reshape(
                mesh.cell_count, 2
            ),
        ),
        eqiora.InitialField(
            plan.capability.pressure,
            vertex_values=np.asarray(steady_pressure.values("vertex")),
        ),
    ),
)
state

In [ ]:
result = eqiora.run(
    plan,
    state=state,
    steps=10,
    output_steps=tuple(range(1, 11)),
    profile=True,
)
print(result.profile.summary())
accepted = result.trajectory.state(10)
vorticity = accepted.curl(plan.capability.velocity)
cylinder_force = accepted.boundary_force(geometry.selection("cylinder"))
front_pressure = accepted.sample(plan.capability.pressure, at=(0.15, 0.2))
rear_pressure = accepted.sample(plan.capability.pressure, at=(0.25, 0.2))
vorticity_values = vorticity.values("cell")

In [ ]:
{
    "status": "UNVERIFIED PRODUCT EXAMPLE — no benchmark acceptance is claimed",
    "geometry": geometry.digest,
    "mesh_plan": mesh_plan.source_digest,
    "mesh": mesh.digest,
    "model": model.digest,
    "plan": plan.identity,
    "trajectory": result.trajectory.digest,
    "accepted_step": accepted.step,
    "accepted_time_s": accepted.time_s,
    "vorticity_unit": "s^-1",
    "vorticity_range": (
        float(vorticity_values.min()),
        float(vorticity_values.max()),
    ),
    "force_on_cylinder_N_per_m": cylinder_force.on_selection,
    "pressure_probes_Pa": (front_pressure.value, rear_pressure.value),
    "pressure_difference_Pa": front_pressure.value - rear_pressure.value,
}

In [ ]:
vorticity_figure = eqplot.plot_scalar_field(
    result.trajectory,
    step=10,
    field=vorticity,
)
vorticity_figure